# Task 4 — Style-Conditioned Face-to-Sketch Generation (Conditional GAN, FS2K)

$$\hat y = G(x,s),\qquad \mathcal{L}_D=\tfrac12\big[\mathrm{BCE}(D(x,y,s),1)+\mathrm{BCE}(D(x,\hat y,s),0)\big],\qquad \mathcal{L}_G=\mathrm{BCE}(D(x,\hat y,s),1)+\lambda_{L1}\,\lVert y-\hat y\rVert_1$$

* **Generator:** a U-Net (pix2pix `unet_128`, 7 downsamplings). A **learned style embedding** is concatenated at the 1×1 bottleneck and drives **conditional instance normalisation** in every decoder block (Dumoulin et al., 2017).
* **Discriminator:** a PatchGAN (14×14 patch logits) that receives (photo, sketch, **its own style-embedding map**), so it judges realism *for the requested style*.
* **Data:** the official FS2K train/test split. 15% of train is held out for validation (seed 42, **stratified by style**), and the test set is used only once, at the end. Images are 128×128.
  Augmentation (random 142→128 crop and horizontal flip) is applied **identically to both images of a pair**.

## ⚠️ Run in this order
1. **`MODE = "smoke"`** (the default) → *Run All*. It takes about 5–10 min and ends with `SMOKE TEST PASSED`. Check the W&B group `task4-smoke`.
2. **`MODE = "full"`** → **Save Version → Save & Run All**. It takes about 2–2.5 h; the runs appear live in the W&B group `task4`.

**Kaggle settings (checked by the pre-flight gate):** GPU **T4** · Internet **On** (W&B, plus the LPIPS / FID network weights) · Secret `WANDB_API_KEY` ticked · Input: the **FS2K** dataset (official zip).

| Section | Content |
|---|---|
| 0–1 | Setup, W&B, FS2K loading + split + **pre-flight gate** |
| 2 | Unit tests (pair-consistent augmentation, shapes, style effect, losses, ONNX with style input) |
| 3 | Optuna (G lr, D lr, batch size, base channels, dropout, style-embedding dim, λ_L1) with shorter trials |
| 4 | Full retraining. D real / D fake / G adversarial / G L1 losses and validation metrics are logged **separately**, and fixed validation photos are logged at fixed intervals |
| 5 | Test evaluation: L1, PSNR, SSIM, LPIPS, FID, overall and per style; examples, style-swap grid, failure cases |
| 6 | Style consistency: a style classifier trained on real sketches checks whether generated sketches carry the requested style |
| 7 | ONNX export of the generator (photo + style → sketch), parity, latency |
| 8 | Summary |

## 0. Setup

In [ ]:
import importlib.util, subprocess, sys
need = {"onnx": "onnx", "onnxruntime": "onnxruntime", "optuna": "optuna", "sklearn": "scikit-learn", "lpips": "lpips",
        "torchmetrics": "torchmetrics", "torch_fidelity": "torch-fidelity"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
print("installed:", missing or "nothing needed")

In [ ]:
%%writefile restoration.py
"""
restoration.py — shared restoration components (Task 1, reused by Task 2 specialists and Task 3).

Contents
  * SSIM (Gaussian window, Wang et al. 2004) + PSNR, per-image and batched, differentiable
  * L1 + SSIM loss:  L = alpha * L1 + (1 - alpha) * (1 - SSIM)
  * ConvAutoencoder: conv encoder -> compressed spatial latent (1x1-conv bottleneck + dropout)
                     -> conv decoder, with optional LIMITED skip connections (deepest levels only)
  * train_one_epoch / evaluate (AMP-aware, per-corruption metrics)
  * ONNX export, PyTorch-vs-ONNX parity check, latency measurement
  * plotting helpers for restoration grids (target / input / output / |error|)
"""
from __future__ import annotations

import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

CLASS_NAMES = ("clean", "salt_pepper", "gaussian_blur", "occlusion")


# ======================================================================================
# Metrics
# ======================================================================================
def _gaussian_window(size: int, sigma: float, channels: int, device, dtype):
    x = torch.arange(size, device=device, dtype=dtype) - (size - 1) / 2
    g = torch.exp(-(x ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    w2d = g[:, None] * g[None, :]
    return w2d.expand(channels, 1, size, size).contiguous()


def ssim_map(x: torch.Tensor, y: torch.Tensor, window_size: int = 11, sigma: float = 1.5,
             data_range: float = 1.0) -> torch.Tensor:
    """SSIM map (B,C,H',W') with an 11x11 Gaussian window (sigma 1.5), 'valid' region only.
    Matches skimage.structural_similarity(gaussian_weights=True, sigma=1.5,
    use_sample_covariance=False, data_range=1)."""
    x, y = x.float(), y.float()
    c = x.shape[1]
    w = _gaussian_window(window_size, sigma, c, x.device, x.dtype)
    c1, c2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    mu_x = F.conv2d(x, w, groups=c)
    mu_y = F.conv2d(y, w, groups=c)
    sxx = F.conv2d(x * x, w, groups=c) - mu_x ** 2
    syy = F.conv2d(y * y, w, groups=c) - mu_y ** 2
    sxy = F.conv2d(x * y, w, groups=c) - mu_x * mu_y
    return ((2 * mu_x * mu_y + c1) * (2 * sxy + c2)) / ((mu_x ** 2 + mu_y ** 2 + c1) * (sxx + syy + c2))


def ssim_per_image(x, y, **kw) -> torch.Tensor:
    return ssim_map(x, y, **kw).mean(dim=(1, 2, 3))


def psnr_per_image(x, y, data_range: float = 1.0) -> torch.Tensor:
    mse = ((x.float() - y.float()) ** 2).mean(dim=(1, 2, 3)).clamp_min(1e-10)
    return 10 * torch.log10(data_range ** 2 / mse)


def mae_per_image(x, y) -> torch.Tensor:
    return (x.float() - y.float()).abs().mean(dim=(1, 2, 3))


def val_score(psnr_mean: float, ssim_mean: float) -> float:
    """Optuna objective (alpha-independent, so the study cannot 'win' by changing the loss scale):
    equal-weight combination of structural similarity and reconstruction fidelity (PSNR/40, capped)."""
    return 0.5 * float(ssim_mean) + 0.5 * min(float(psnr_mean) / 40.0, 1.0)


# ======================================================================================
# Loss
# ======================================================================================
class L1SSIMLoss(nn.Module):
    """alpha * L1 + (1 - alpha) * (1 - SSIM). Always computed in float32 (AMP-safe)."""

    def __init__(self, alpha: float = 0.8):
        super().__init__()
        self.alpha = float(alpha)

    def forward(self, pred, target):
        pred, target = pred.float(), target.float()
        l1 = F.l1_loss(pred, target)
        s = ssim_map(pred, target).mean()
        total = self.alpha * l1 + (1 - self.alpha) * (1 - s)
        return total, {"l1": l1.detach(), "ssim": s.detach()}


# ======================================================================================
# Model
# ======================================================================================
def norm(c: int) -> nn.GroupNorm:
    # GroupNorm instead of BatchNorm: no running statistics, so train- and eval-mode behave the same
    # at any batch size (Optuna samples batch sizes 16-64), and frozen experts in Task 3 cannot drift.
    return nn.GroupNorm(min(8, c), c)


def conv_block(cin: int, cout: int, stride: int = 1) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, stride, 1, bias=False), norm(cout), nn.SiLU(inplace=True),
        nn.Conv2d(cout, cout, 3, 1, 1, bias=False), norm(cout), nn.SiLU(inplace=True),
    )


class ConvAutoencoder(nn.Module):
    """Convolutional denoising autoencoder with a genuine compressed code.

    Encoder: 128 -> 64 -> 32 -> 16 -> 8 (stride-2 conv blocks), channels grow base x (1,2,4,8,8).
    Latent:  1x1 conv to `latent_channels` at 8x8 (e.g. 16 ch -> 1,024 values), then code dropout.
    Decoder: bilinear upsample + conv blocks (no transposed-conv checkerboard artefacts), sigmoid output.

    Skip connections (skip_levels = 0, 1 or 2) are BOTTLENECKED: the encoder feature at the
    16x16 (and 32x32) level is squeezed by a 1x1 conv to `skip_channels` channels and passes the same
    code dropout before reaching the decoder. The information the decoder receives is therefore
    bounded by the total code size

        code_dim = latent_dim + sum_over_skipped_levels(skip_channels * H_l * W_l),

    which is what `compression_ratio` reports (input values / code values). Full- and half-resolution
    features are never skipped. skip_channels=None reproduces the original full-width skips
    (e.g. 384 x 16 x 16 = 98,304 values, larger than the image); it is kept only for the ablation,
    which showed that such skips let the decoder ignore the latent.
    """

    def __init__(self, base_channels: int = 32, latent_channels: int = 32, dropout: float = 0.1,
                 skip_levels: int = 0, skip_channels: int | None = 4, depth: int = 4, max_mult: int = 8,
                 img_size: int = 128):
        super().__init__()
        assert 0 <= skip_levels <= 2, "only the two deepest levels may be skipped"
        self.config = dict(base_channels=base_channels, latent_channels=latent_channels, dropout=dropout,
                           skip_levels=skip_levels, skip_channels=skip_channels, depth=depth,
                           max_mult=max_mult, img_size=img_size)
        chs = [min(base_channels * 2 ** i, base_channels * max_mult) for i in range(depth + 1)]
        self.chs, self.depth, self.skip_levels, self.img_size = chs, depth, skip_levels, img_size
        self.stem = conv_block(3, chs[0], 1)
        self.down = nn.ModuleList([conv_block(chs[i], chs[i + 1], 2) for i in range(depth)])
        self.to_latent = nn.Conv2d(chs[-1], latent_channels, 1)
        self.code_dropout = nn.Dropout(dropout)
        self.from_latent = nn.Sequential(nn.Conv2d(latent_channels, chs[-1], 1, bias=False),
                                         norm(chs[-1]), nn.SiLU(inplace=True))
        # decoder stage j upsamples to resolution level (depth - j - 1); skips only for j < skip_levels
        self.skip_proj = nn.ModuleList()
        self.skip_shapes = []
        self.up = nn.ModuleList()
        for j in range(depth):
            lvl = depth - j - 1
            extra = 0
            if j < skip_levels:
                width = chs[lvl] if skip_channels is None else skip_channels
                self.skip_proj.append(nn.Identity() if skip_channels is None else nn.Conv2d(chs[lvl], skip_channels, 1))
                res = img_size // 2 ** lvl
                self.skip_shapes.append((width, res, res))
                extra = width
            self.up.append(conv_block(chs[lvl + 1] + extra, chs[lvl], 1))
        self.head = nn.Conv2d(chs[0], 3, 3, 1, 1)
        self.latent_shape = (latent_channels, img_size // 2 ** depth, img_size // 2 ** depth)

    # ---- code-size accounting ------------------------------------------------------------
    @property
    def latent_dim(self) -> int:
        return int(np.prod(self.latent_shape))

    @property
    def skip_dim(self) -> int:
        return int(sum(np.prod(s) for s in self.skip_shapes))

    @property
    def code_dim(self) -> int:
        """Total number of values the decoder receives from the encoder (latent + skip codes)."""
        return self.latent_dim + self.skip_dim

    @property
    def compression_ratio(self) -> float:
        return 3 * self.img_size ** 2 / self.code_dim

    # ---- forward -------------------------------------------------------------------------
    def encode(self, x):
        """Returns (latent, skip_codes). skip_codes[j] feeds decoder stage j (deepest first)."""
        feats = []
        h = self.stem(x)
        for blk in self.down:
            feats.append(h)
            h = blk(h)
        skips = [proj(feats[self.depth - j - 1]) for j, proj in enumerate(self.skip_proj)]
        return self.to_latent(h), skips

    def decode(self, z, skips=None):
        h = self.from_latent(self.code_dropout(z))
        for j, blk in enumerate(self.up):
            h = F.interpolate(h, scale_factor=2, mode="bilinear", align_corners=False)
            if j < self.skip_levels:
                s = skips[j] if skips is not None else h.new_zeros(h.shape[0], *self.skip_shapes[j])
                h = torch.cat([h, self.code_dropout(s)], dim=1)
            h = blk(h)
        return torch.sigmoid(self.head(h))

    def forward(self, x):
        z, skips = self.encode(x)
        return self.decode(z, skips)


@torch.no_grad()
def code_dependence(model, loader, device, max_batches=None) -> dict:
    """How much the output depends on each part of the code (evaluated on a manifest loader).

    full:        normal forward pass
    zero_latent: latent replaced by zeros (skips intact)   -> large drop = the latent is used
    zero_skips:  skip codes replaced by zeros (latent intact) -> drop = what the skips contribute
    """
    model.eval()
    acc = {k: {"psnr": [], "ssim": []} for k in ("full", "zero_latent", "zero_skips")}
    for b, batch in enumerate(loader):
        if max_batches is not None and b >= max_batches:
            break
        x, y = batch[0].to(device), batch[1].to(device)
        z, skips = model.encode(x)
        outs = {"full": model.decode(z, skips), "zero_latent": model.decode(torch.zeros_like(z), skips)}
        if model.skip_levels:
            outs["zero_skips"] = model.decode(z, [torch.zeros_like(s) for s in skips])
        for k, o in outs.items():
            o = o.float().clamp(0, 1)
            acc[k]["psnr"].append(psnr_per_image(o, y).cpu())
            acc[k]["ssim"].append(ssim_per_image(o, y).cpu())
    out = {}
    for k, d in acc.items():
        if d["psnr"]:
            out[f"{k}_psnr"] = torch.cat(d["psnr"]).mean().item()
            out[f"{k}_ssim"] = torch.cat(d["ssim"]).mean().item()
    out["latent_drop_db"] = out["full_psnr"] - out["zero_latent_psnr"]
    if "zero_skips_psnr" in out:
        out["skip_drop_db"] = out["full_psnr"] - out["zero_skips_psnr"]
    return out


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ======================================================================================
# Training / evaluation
# ======================================================================================
def make_optimizer_and_scheduler(model, lr, weight_decay, steps_per_epoch, epochs, warmup_epochs=1):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total = max(1, steps_per_epoch * epochs)
    warm = max(1, int(steps_per_epoch * warmup_epochs))

    def lr_lambda(step):
        if step < warm:
            return (step + 1) / warm
        t = (step - warm) / max(1, total - warm)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(1.0, t)))

    return opt, torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)


def make_scaler(device, amp=True):
    """One GradScaler per training run (fp16 AMP on CUDA only; a no-op on CPU)."""
    return torch.amp.GradScaler("cuda", enabled=bool(amp and device.type == "cuda"))


def train_one_epoch(model, loader, loss_fn, optimizer, scheduler, device, scaler=None, amp=True,
                    grad_clip=1.0, max_batches=None) -> dict:
    model.train()
    use_amp = amp and device.type == "cuda"
    if scaler is None:
        scaler = make_scaler(device, amp)
    tot = {"loss": 0.0, "l1": 0.0, "ssim": 0.0}
    n = 0
    for b, batch in enumerate(loader):
        if max_batches is not None and b >= max_batches:
            break
        x, y = batch[0].to(device, non_blocking=True), batch[1].to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            pred = model(x)
        loss, parts = loss_fn(pred, y)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"non-finite loss at batch {b}")
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        if grad_clip:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        bs = x.shape[0]
        tot["loss"] += loss.item() * bs
        tot["l1"] += parts["l1"].item() * bs
        tot["ssim"] += parts["ssim"].item() * bs
        n += bs
    return {k: v / max(n, 1) for k, v in tot.items()}


@torch.no_grad()
def evaluate(model, loader, device, loss_fn=None, amp=True, per_image=False, include_input=False) -> dict:
    """Metrics over a (manifest) loader. Returns means overall and per corruption type.
    per_image=True also returns numpy arrays aligned with the dataset order.
    include_input=True also scores the corrupted input itself (the 'do nothing' baseline)."""
    model.eval()
    use_amp = amp and device.type == "cuda"
    keys = ["psnr", "ssim", "mae"] + (["in_psnr", "in_ssim", "in_mae"] if include_input else [])
    acc = {k: [] for k in keys}
    labels, losses = [], []
    for batch in loader:
        x, y, lab = batch[0].to(device), batch[1].to(device), batch[2]
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            pred = model(x)
        pred = pred.float().clamp(0, 1)
        if loss_fn is not None:
            losses.append(loss_fn(pred, y)[0].item() * x.shape[0])
        acc["psnr"].append(psnr_per_image(pred, y).cpu())
        acc["ssim"].append(ssim_per_image(pred, y).cpu())
        acc["mae"].append(mae_per_image(pred, y).cpu())
        if include_input:
            acc["in_psnr"].append(psnr_per_image(x, y).cpu())
            acc["in_ssim"].append(ssim_per_image(x, y).cpu())
            acc["in_mae"].append(mae_per_image(x, y).cpu())
        labels.append(lab.cpu())
    arr = {k: torch.cat(v).numpy() for k, v in acc.items()}
    labels = torch.cat(labels).numpy()
    out = {k: float(v.mean()) for k, v in arr.items()}
    for c, name in enumerate(CLASS_NAMES):
        m = labels == c
        if m.any():
            for k, v in arr.items():
                out[f"{k}/{name}"] = float(v[m].mean())
    out["score"] = val_score(out["psnr"], out["ssim"])
    if loss_fn is not None:
        out["loss"] = sum(losses) / len(labels)
    if per_image:
        out["per_image"] = {**arr, "label": labels}
    return out


@torch.no_grad()
def predict_numpy(model, imgs_hwc: np.ndarray, device, batch_size=64) -> np.ndarray:
    """(N,H,W,3) float [0,1] -> restored (N,H,W,3) float [0,1]."""
    model.eval()
    outs = []
    for i in range(0, len(imgs_hwc), batch_size):
        x = torch.from_numpy(np.ascontiguousarray(imgs_hwc[i:i + batch_size].transpose(0, 3, 1, 2))).float().to(device)
        outs.append(model(x).float().clamp(0, 1).cpu().numpy().transpose(0, 2, 3, 1))
    return np.concatenate(outs)


# ======================================================================================
# ONNX
# ======================================================================================
def export_onnx(model, path, img_size=128, opset=17, input_name="input", output_name="output"):
    model = model.eval().cpu()
    dummy = torch.rand(1, 3, img_size, img_size)
    kw = dict(input_names=[input_name], output_names=[output_name], opset_version=opset,
              dynamic_axes={input_name: {0: "batch"}, output_name: {0: "batch"}})
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with torch.no_grad():
        try:
            torch.onnx.export(model, dummy, str(path), dynamo=False, **kw)
        except TypeError:          # torch < 2.5 has no `dynamo` kwarg
            torch.onnx.export(model, dummy, str(path), **kw)
    try:
        import onnx
        onnx.checker.check_model(onnx.load(str(path)))
    except ImportError:
        pass
    return path


def verify_onnx(model, path, x: torch.Tensor, atol=1e-4) -> dict:
    """Compare PyTorch (CPU, eval) and onnxruntime outputs on the same batch."""
    import onnxruntime as ort
    model = model.eval().cpu()
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    with torch.no_grad():
        ref = model(x.cpu().float()).numpy()
    out = sess.run(None, {sess.get_inputs()[0].name: x.cpu().float().numpy()})[0]
    diff = np.abs(out - ref)
    p_ref = psnr_per_image(torch.from_numpy(ref), torch.from_numpy(out))
    return {"max_abs_diff": float(diff.max()), "mean_abs_diff": float(diff.mean()),
            "min_psnr_torch_vs_onnx": float(p_ref.min()), "n_images": int(len(x)),
            "passed": bool(diff.max() < atol)}


def onnx_latency_ms(path, img_size=128, batch=1, n_warmup=10, n_runs=100) -> dict:
    import onnxruntime as ort
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    name = sess.get_inputs()[0].name
    x = np.random.rand(batch, 3, img_size, img_size).astype(np.float32)
    for _ in range(n_warmup):
        sess.run(None, {name: x})
    ts = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        sess.run(None, {name: x})
        ts.append((time.perf_counter() - t0) * 1000)
    return {"median_ms": float(np.median(ts)), "p90_ms": float(np.percentile(ts, 90)), "batch": batch}


# ======================================================================================
# Plotting
# ======================================================================================
def restoration_grid(rows: list[dict], path=None, title=None, err_vmax=0.5):
    """rows: dicts with 'clean', 'corrupted', 'restored' (HxWx3 float [0,1]) and optional 'label'.
    Columns: clean target | corrupted input | reconstruction | absolute error map."""
    import matplotlib.pyplot as plt
    n = len(rows)
    fig, axes = plt.subplots(n, 4, figsize=(9.2, 2.35 * n), squeeze=False)
    for i, r in enumerate(rows):
        err = np.abs(r["restored"] - r["clean"]).mean(-1)
        panels = [r["clean"], r["corrupted"], r["restored"]]
        for j, im in enumerate(panels):
            axes[i, j].imshow(np.clip(im, 0, 1))
        hm = axes[i, 3].imshow(err, cmap="magma", vmin=0, vmax=err_vmax)
        for ax in axes[i]:
            ax.set_xticks([]); ax.set_yticks([])
        axes[i, 0].set_ylabel(r.get("label", ""), fontsize=7.5, rotation=0, ha="right", va="center", labelpad=4)
        if "in_text" in r:
            axes[i, 1].set_xlabel(r["in_text"], fontsize=7.5)
        if "out_text" in r:
            axes[i, 2].set_xlabel(r["out_text"], fontsize=7.5)
    for j, t in enumerate(["clean target", "corrupted input", "reconstruction", "|error| (mean RGB)"]):
        axes[0, j].set_title(t, fontsize=9)
    fig.colorbar(hm, ax=axes[:, 3].tolist(), fraction=0.046, pad=0.02, shrink=0.6)
    if title:
        fig.suptitle(title, fontsize=11)
    if path:
        fig.savefig(path, dpi=150, bbox_inches="tight")
    return fig

In [ ]:
%%writefile classifier.py
"""
classifier.py — corruption classifier for Task 2 (reused as the gating network in Task 3).

Classes (same order everywhere, incl. the Task 3 gate weights w0..w3):
    0 = clean, 1 = salt_pepper, 2 = gaussian_blur, 3 = occlusion

Design notes
  * The first conv block runs at FULL resolution (128x128): salt-and-pepper noise and slight blur are
    pixel-level cues that are destroyed by early downsampling.
  * GroupNorm (no running statistics): train/eval behave identically, which matters when the network is
    later fine-tuned as the Task 3 gate with small batches.
  * Global AVERAGE + global MAX pooling are concatenated: max pooling responds to rare extreme activations
    (isolated black/white impulses, hard edges of black occlusion boxes); average pooling to global
    statistics (overall loss of high frequencies under blur).
  * forward() returns LOGITS; softmax / temperature are applied by the caller (Task 3: softmax(G(x)/tau)).
"""
from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

CLASS_NAMES = ("clean", "salt_pepper", "gaussian_blur", "occlusion")
CHANNEL_CONFIGS = {
    "16-32-64-128": (16, 32, 64, 128),
    "32-64-128-256": (32, 64, 128, 256),
    "32-64-128-256-256": (32, 64, 128, 256, 256),
}


def _gn(c):
    return nn.GroupNorm(min(8, c), c)


class CorruptionClassifier(nn.Module):
    def __init__(self, channels=(32, 64, 128, 256), dropout: float = 0.2, num_classes: int = 4):
        super().__init__()
        if isinstance(channels, str):
            channels = CHANNEL_CONFIGS[channels]
        channels = tuple(int(c) for c in channels)
        self.config = dict(channels=list(channels), dropout=dropout, num_classes=num_classes)
        layers, cin = [], 3
        for i, c in enumerate(channels):
            stride = 1 if i == 0 else 2            # stage 0 keeps full resolution
            layers += [nn.Conv2d(cin, c, 3, stride, 1, bias=False), _gn(c), nn.SiLU(inplace=True),
                       nn.Conv2d(c, c, 3, 1, 1, bias=False), _gn(c), nn.SiLU(inplace=True)]
            cin = c
        self.features = nn.Sequential(*layers)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(2 * cin, num_classes)

    def forward(self, x):
        h = self.features(x)
        h = torch.cat([h.mean(dim=(2, 3)), h.amax(dim=(2, 3))], dim=1)
        return self.fc(self.dropout(h))


class ClassifierWithProbs(nn.Module):
    """ONNX wrapper: returns (logits, probabilities)."""

    def __init__(self, clf):
        super().__init__()
        self.clf = clf

    def forward(self, x):
        logits = self.clf(x)
        return logits, torch.softmax(logits, dim=1)


# --------------------------------------------------------------------------------------
# Training / evaluation
# --------------------------------------------------------------------------------------
def train_one_epoch(model, loader, optimizer, scheduler, device, scaler, amp=True, grad_clip=1.0, label_smoothing=0.0):
    model.train()
    use_amp = amp and device.type == "cuda"
    tot_loss, correct, n = 0.0, 0, 0
    for x, _, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(x)
        loss = F.cross_entropy(logits.float(), y, label_smoothing=label_smoothing)
        if not torch.isfinite(loss):
            raise FloatingPointError("non-finite classifier loss")
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        if grad_clip:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        tot_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        n += len(y)
    return {"loss": tot_loss / max(n, 1), "acc": correct / max(n, 1)}


@torch.no_grad()
def predict(model, loader, device, amp=True):
    """Returns (labels, probs[N,4], logits[N,4]) over a manifest loader (dataset order)."""
    model.eval()
    use_amp = amp and device.type == "cuda"
    labels, logits = [], []
    for batch in loader:
        x = batch[0].to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            lg = model(x)
        logits.append(lg.float().cpu())
        labels.append(batch[2])
    logits = torch.cat(logits)
    return torch.cat(labels).numpy(), torch.softmax(logits, 1).numpy(), logits.numpy()


def classification_metrics(y_true, probs) -> dict:
    """Accuracy, macro P/R/F1, per-class P/R/F1/support, row-normalised confusion matrix, CE loss."""
    from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
    y_pred = probs.argmax(1)
    labels = list(range(len(CLASS_NAMES)))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    mp, mr, mf, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ce = float(-np.log(np.clip(probs[np.arange(len(y_true)), y_true], 1e-12, 1)).mean())
    out = {"acc": float(accuracy_score(y_true, y_pred)), "macro_precision": float(mp), "macro_recall": float(mr),
           "macro_f1": float(mf), "loss": ce, "cm": cm, "cm_norm": cm_norm}
    for i, n in enumerate(CLASS_NAMES):
        out[f"precision/{n}"], out[f"recall/{n}"], out[f"f1/{n}"], out[f"support/{n}"] = float(p[i]), float(r[i]), float(f[i]), int(s[i])
    return out


def per_class_table(m: dict):
    import pandas as pd
    rows = [{"class": n, "precision": m[f"precision/{n}"], "recall": m[f"recall/{n}"], "f1": m[f"f1/{n}"],
             "support": m[f"support/{n}"]} for n in CLASS_NAMES]
    rows.append({"class": "macro avg", "precision": m["macro_precision"], "recall": m["macro_recall"],
                 "f1": m["macro_f1"], "support": int(sum(m[f"support/{n}"] for n in CLASS_NAMES))})
    rows.append({"class": "accuracy", "precision": np.nan, "recall": np.nan, "f1": m["acc"],
                 "support": int(sum(m[f"support/{n}"] for n in CLASS_NAMES))})
    return pd.DataFrame(rows).set_index("class")


def plot_confusion(cm_norm, cm_counts=None, title="", path=None, ax=None):
    import matplotlib.pyplot as plt
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(4.8, 4.2))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    short = ["clean", "s&p", "blur", "occl."]
    ax.set_xticks(range(4)); ax.set_xticklabels(short); ax.set_yticks(range(4)); ax.set_yticklabels(short)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    for i in range(4):
        for j in range(4):
            txt = f"{cm_norm[i, j]:.2f}" + (f"\n({cm_counts[i, j]})" if cm_counts is not None else "")
            ax.text(j, i, txt, ha="center", va="center", fontsize=8,
                    color="white" if cm_norm[i, j] > 0.6 else "black")
    ax.set_title(title, fontsize=10)
    if own:
        fig.colorbar(im, ax=ax, fraction=0.046)
        fig.tight_layout()
        if path:
            fig.savefig(path, dpi=150)
        return fig
    return im

In [ ]:
%%writefile sketch_gan.py
"""
sketch_gan.py — Task 4: style-conditioned face-to-sketch generation with a conditional GAN (FS2K).

    y_hat = G(x, s)            x: photo, s in {0,1,2}: sketch style (FS2K "style" field)
    D(x, y, s) -> patch logits (PatchGAN)
    L_D = 0.5 * [BCE(D(x,y,s), 1) + BCE(D(x,G(x,s),s), 0)]
    L_G = BCE(D(x,G(x,s),s), 1) + lambda_L1 * L1(y, G(x,s))

Style conditioning (learned embeddings, separate tables in G and D):
  * Generator: the style embedding is concatenated to the 1x1 bottleneck AND drives conditional instance
    normalisation (per-style gamma/beta) in every decoder block, the multi-style mechanism of
    Dumoulin et al. 2017 ("A learned representation for artistic style").
  * Discriminator: its own embedding is broadcast to a spatial map and concatenated to (photo, sketch), so D
    judges whether a sketch is real FOR THAT STYLE.
Images are in [-1, 1] inside the networks; the ONNX wrapper takes/returns [0, 1].
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

STYLE_NAMES = ("Style 1", "Style 2", "Style 3")
IMG = 128
JITTER = 142          # train images are stored at 142x142 and randomly cropped to 128 (pix2pix-style jitter)


# --------------------------------------------------------------------------------------
# Dataset discovery / loading
# --------------------------------------------------------------------------------------
def find_fs2k(search_root="/kaggle/input") -> Path:
    """Folder that contains anno_train.json, anno_test.json, photo/ and sketch/."""
    for p in sorted(Path(search_root).rglob("anno_train.json")):
        r = p.parent
        if (r / "anno_test.json").exists() and (r / "photo").is_dir() and (r / "sketch").is_dir():
            return r
    raise FileNotFoundError(f"FS2K not found under {search_root}: need anno_train.json, anno_test.json, photo/, sketch/")


def _with_ext(base: Path) -> Path | None:
    for ext in (".jpg", ".png", ".jpeg", ".JPG", ".PNG"):
        if base.with_suffix(ext).exists():
            return base.with_suffix(ext)
    return None


def read_annotations(root: Path, split: str) -> list[dict]:
    """[{name, photo, sketch, style}] for the official split. Mapping follows FS2K's own split script:
    photo/photoK/imageNNNN.{jpg,png}  <->  sketch/sketchK/sketchNNNN.{jpg,png}"""
    entries = json.loads((root / f"anno_{split}.json").read_text())
    out, missing = [], []
    for e in entries:
        name = e["image_name"]
        photo = _with_ext(root / "photo" / name)
        sketch = _with_ext(root / "sketch" / name.replace("image", "sketch").replace("photo", "sketch"))
        if photo is None or sketch is None:
            missing.append(name); continue
        out.append({"name": name, "photo": str(photo), "sketch": str(sketch), "style": int(e["style"])})
    if missing:
        print(f"[{split}] {len(missing)} entries without photo/sketch file, e.g. {missing[:3]}")
    return out


def stratified_val_split(records, frac=0.15, seed=42):
    from sklearn.model_selection import train_test_split
    idx = np.arange(len(records))
    tr, va = train_test_split(idx, test_size=frac, random_state=seed, stratify=[r["style"] for r in records])
    return [records[i] for i in sorted(tr)], [records[i] for i in sorted(va)]


def load_rgb(path, size):
    with Image.open(path) as im:
        if im.mode in ("RGBA", "LA", "P"):
            im = im.convert("RGBA")
            bg = Image.new("RGBA", im.size, (255, 255, 255, 255))       # transparent -> white (sketch paper)
            im = Image.alpha_composite(bg, im)
        return np.asarray(im.convert("RGB").resize((size, size), Image.Resampling.BICUBIC), dtype=np.uint8).copy()


def load_pairs(records, size, cache=None):
    """Both members of a pair are resized identically -> pixel correspondence preserved."""
    if cache is not None and Path(cache).exists():
        z = np.load(cache)
        if list(z["names"]) == [r["name"] for r in records] and int(z["size"]) == size:
            return z["photos"], z["sketches"]
    photos = np.stack([load_rgb(r["photo"], size) for r in records])
    sketches = np.stack([load_rgb(r["sketch"], size) for r in records])
    if cache is not None:
        Path(cache).parent.mkdir(parents=True, exist_ok=True)
        np.savez(cache, photos=photos, sketches=sketches, names=np.array([r["name"] for r in records]), size=size)
    return photos, sketches


class PairDataset(torch.utils.data.Dataset):
    """Returns (photo, sketch, style) in [-1,1]. With augment=True the SAME random crop and flip are applied to
    both images of the pair (spatial correspondence must be preserved for paired training)."""

    def __init__(self, photos, sketches, styles, augment=False, size=IMG):
        self.p, self.s, self.st, self.aug, self.size = photos, sketches, np.asarray(styles), augment, size

    def __len__(self):
        return len(self.p)

    def __getitem__(self, i):
        p, s = self.p[i], self.s[i]
        if self.aug:
            H = p.shape[0]
            y0, x0 = np.random.randint(0, H - self.size + 1, size=2)
            p, s = p[y0:y0 + self.size, x0:x0 + self.size], s[y0:y0 + self.size, x0:x0 + self.size]
            if np.random.rand() < 0.5:
                p, s = p[:, ::-1], s[:, ::-1]
        t = lambda a: torch.from_numpy(np.ascontiguousarray(a.transpose(2, 0, 1))).float() / 127.5 - 1.0
        return t(p), t(s), torch.tensor(int(self.st[i]))


# --------------------------------------------------------------------------------------
# Models
# --------------------------------------------------------------------------------------
class CondInstanceNorm(nn.Module):
    """InstanceNorm whose scale/shift are predicted from the style embedding (conditional IN)."""

    def __init__(self, ch, emb_dim):
        super().__init__()
        self.norm = nn.InstanceNorm2d(ch, affine=False)
        self.gamma = nn.Linear(emb_dim, ch)
        self.beta = nn.Linear(emb_dim, ch)
        nn.init.zeros_(self.gamma.weight); nn.init.zeros_(self.gamma.bias)
        nn.init.zeros_(self.beta.weight); nn.init.zeros_(self.beta.bias)

    def forward(self, x, e):
        return self.norm(x) * (1 + self.gamma(e)[:, :, None, None]) + self.beta(e)[:, :, None, None]


class UNetGenerator(nn.Module):
    """pix2pix 'unet_128' (7 downsamplings 128 -> 1) with style conditioning."""

    def __init__(self, base_channels=64, emb_dim=16, dropout=0.5, n_styles=3):
        super().__init__()
        self.config = dict(base_channels=base_channels, emb_dim=emb_dim, dropout=dropout, n_styles=n_styles)
        g = base_channels
        self.style_emb = nn.Embedding(n_styles, emb_dim)
        enc = [g, 2 * g, 4 * g, 8 * g, 8 * g, 8 * g, 8 * g]
        self.down = nn.ModuleList()
        cin = 3
        for i, c in enumerate(enc):
            layers = [] if i == 0 else [nn.LeakyReLU(0.2, inplace=True)]
            layers.append(nn.Conv2d(cin, c, 4, 2, 1, bias=False))
            if 0 < i < len(enc) - 1:
                layers.append(nn.InstanceNorm2d(c, affine=True))
            self.down.append(nn.Sequential(*layers))
            cin = c
        dec_out = [8 * g, 8 * g, 8 * g, 4 * g, 2 * g, g]           # u7 .. u2
        skips = enc[::-1][1:]                                        # d6 .. d1 channels
        self.up_conv, self.up_norm, self.up_drop = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        cin = enc[-1] + emb_dim                                      # bottleneck + style embedding
        for i, c in enumerate(dec_out):
            self.up_conv.append(nn.ConvTranspose2d(cin, c, 4, 2, 1, bias=False))
            self.up_norm.append(CondInstanceNorm(c, emb_dim))
            self.up_drop.append(nn.Dropout(dropout if i < 3 else 0.0))
            cin = c + skips[i]
        self.final = nn.ConvTranspose2d(cin, 3, 4, 2, 1)

    def forward(self, x, style):
        e = self.style_emb(style)
        feats, h = [], x
        for d in self.down:
            h = d(h)
            feats.append(h)
        h = torch.cat([feats[-1], e[:, :, None, None].expand(-1, -1, h.shape[2], h.shape[3])], 1)
        for i in range(len(self.up_conv)):
            h = self.up_conv[i](F.relu(h))
            h = self.up_drop[i](self.up_norm[i](h, e))
            h = torch.cat([h, feats[-2 - i]], 1)
        return torch.tanh(self.final(F.relu(h)))


class PatchDiscriminator(nn.Module):
    """70x70-style PatchGAN on (photo, sketch, style map) -> 14x14 patch logits for 128x128 inputs."""

    def __init__(self, base_channels=64, emb_dim=16, n_styles=3):
        super().__init__()
        self.config = dict(base_channels=base_channels, emb_dim=emb_dim, n_styles=n_styles)
        d = base_channels
        self.style_emb = nn.Embedding(n_styles, emb_dim)
        self.net = nn.Sequential(
            nn.Conv2d(6 + emb_dim, d, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(d, 2 * d, 4, 2, 1, bias=False), nn.InstanceNorm2d(2 * d, affine=True), nn.LeakyReLU(0.2, True),
            nn.Conv2d(2 * d, 4 * d, 4, 2, 1, bias=False), nn.InstanceNorm2d(4 * d, affine=True), nn.LeakyReLU(0.2, True),
            nn.Conv2d(4 * d, 8 * d, 4, 1, 1, bias=False), nn.InstanceNorm2d(8 * d, affine=True), nn.LeakyReLU(0.2, True),
            nn.Conv2d(8 * d, 1, 4, 1, 1))

    def forward(self, photo, sketch, style):
        e = self.style_emb(style)[:, :, None, None].expand(-1, -1, photo.shape[2], photo.shape[3])
        return self.net(torch.cat([photo, sketch, e], 1))


def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.InstanceNorm2d) and m.affine:
        nn.init.normal_(m.weight, 1.0, 0.02); nn.init.zeros_(m.bias)


class GeneratorExport(nn.Module):
    """ONNX wrapper: photo [B,3,128,128] in [0,1] + style int64 [B] (0,1,2) -> sketch [B,3,128,128] in [0,1]."""

    def __init__(self, G):
        super().__init__()
        self.G = G

    def forward(self, photo, style):
        return (self.G(photo * 2 - 1, style) + 1) / 2


# --------------------------------------------------------------------------------------
# Training
# --------------------------------------------------------------------------------------
def train_one_epoch(G, D, loader, opt_g, opt_d, device, lambda_l1, scaler_g, scaler_d, amp=True):
    G.train(); D.train()
    use_amp = amp and device.type == "cuda"
    bce = nn.BCEWithLogitsLoss()
    agg = {k: 0.0 for k in ("d_real", "d_fake", "d_loss", "g_adv", "g_l1", "g_loss", "d_acc_real", "d_acc_fake")}
    n = 0
    for x, y, s in loader:
        x, y, s = x.to(device, non_blocking=True), y.to(device, non_blocking=True), s.to(device, non_blocking=True)
        # ---- discriminator
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            fake = G(x, s)
            lr_real = D(x, y, s)
            lr_fake = D(x, fake.detach(), s)
        d_real = bce(lr_real.float(), torch.ones_like(lr_real, dtype=torch.float32))
        d_fake = bce(lr_fake.float(), torch.zeros_like(lr_fake, dtype=torch.float32))
        d_loss = 0.5 * (d_real + d_fake)
        opt_d.zero_grad(set_to_none=True)
        scaler_d.scale(d_loss).backward(); scaler_d.step(opt_d); scaler_d.update()
        # ---- generator
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            lg_fake = D(x, fake, s)
        g_adv = bce(lg_fake.float(), torch.ones_like(lg_fake, dtype=torch.float32))
        g_l1 = F.l1_loss(fake.float(), y.float())
        g_loss = g_adv + lambda_l1 * g_l1
        if not torch.isfinite(g_loss) or not torch.isfinite(d_loss):
            raise FloatingPointError("non-finite GAN loss")
        opt_g.zero_grad(set_to_none=True)
        scaler_g.scale(g_loss).backward(); scaler_g.step(opt_g); scaler_g.update()
        b = len(s)
        for k, v in (("d_real", d_real), ("d_fake", d_fake), ("d_loss", d_loss), ("g_adv", g_adv), ("g_l1", g_l1), ("g_loss", g_loss)):
            agg[k] += v.item() * b
        agg["d_acc_real"] += (lr_real.float() > 0).float().mean().item() * b
        agg["d_acc_fake"] += (lr_fake.float() < 0).float().mean().item() * b
        n += b
    return {k: v / max(n, 1) for k, v in agg.items()}


def linear_decay(total_epochs):
    """pix2pix schedule: constant lr for the first half, then linear decay to 0."""
    half = total_epochs // 2
    return lambda ep: 1.0 if ep < half else max(0.0, 1.0 - (ep - half) / max(1, total_epochs - half))


@torch.no_grad()
def generate(G, photos_m11, styles, device, bs=64):
    """photos in [-1,1] (N,3,H,W) tensor, styles (N,) -> sketches in [0,1] numpy (N,3,H,W)."""
    G.eval()
    outs = []
    for i in range(0, len(photos_m11), bs):
        o = G(photos_m11[i:i + bs].to(device), styles[i:i + bs].to(device))
        outs.append(((o.float() + 1) / 2).clamp(0, 1).cpu())
    return torch.cat(outs)


class Metrics:
    """L1 / PSNR / SSIM always; LPIPS if the lpips package + AlexNet weights are available."""

    def __init__(self, device):
        import restoration as R
        self.R, self.device, self.lpips = R, device, None
        try:
            import lpips
            self.lpips = lpips.LPIPS(net="alex", verbose=False).to(device).eval()
        except Exception as e:
            self.lpips_error = repr(e)

    @torch.no_grad()
    def per_image(self, fake01, real01):
        R = self.R
        out = {"l1": (fake01 - real01).abs().mean(dim=(1, 2, 3)).numpy(),
               "psnr": R.psnr_per_image(fake01, real01).numpy(), "ssim": R.ssim_per_image(fake01, real01).numpy()}
        if self.lpips is not None:
            vals = []
            for i in range(0, len(fake01), 64):
                a = fake01[i:i + 64].to(self.device) * 2 - 1; b = real01[i:i + 64].to(self.device) * 2 - 1
                vals.append(self.lpips(a, b).flatten().cpu())
            out["lpips"] = torch.cat(vals).numpy()
        return out


def val_score(m: dict) -> float:
    """Optuna objective: structure (SSIM) + perceptual similarity (1 - LPIPS); SSIM only if LPIPS is unavailable."""
    if "lpips" in m:
        return float(0.5 * np.mean(m["ssim"]) + 0.5 * (1 - np.mean(m["lpips"])))
    return float(np.mean(m["ssim"]))

In [ ]:
import os, sys, json, time, math, shutil, importlib, warnings, gc, random
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import optuna

sys.path.insert(0, str(Path.cwd()))
import restoration as R, classifier as C, sketch_gan as SG
for m_ in (R, C, SG):
    importlib.reload(m_)
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)
warnings.filterwarnings("ignore", message=".*lr_scheduler.step.*")
warnings.filterwarnings("ignore", message=".*instance_norm.*train=True.*")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ============================ MAIN SWITCH ============================
MODE = "smoke"          # "smoke" first, then "full"
# =====================================================================
MODE = os.environ.get("T4_MODE", MODE)
SETTINGS = {
    "smoke": dict(n_trials=3, trial_epochs=2, optuna_timeout=None, final_epochs=3, sample_every=1, style_epochs=2,
                  train_subset=None, eval_subset=None),
    "full":  dict(n_trials=20, trial_epochs=30, optuna_timeout=int(1.5 * 3600), final_epochs=200, sample_every=10, style_epochs=20,
                  train_subset=None, eval_subset=None),
}
S = dict(SETTINGS[MODE])
S.update(json.loads(os.environ.get("T4_SETTINGS_OVERRIDE", "{}")))   # local CPU testing only

IS_KAGGLE  = Path("/kaggle/input").exists()
INPUT_ROOT = Path(os.environ.get("FS2K_INPUT_ROOT", "/kaggle/input"))
WORK_DIR   = Path(os.environ.get("WORK_DIR", "/kaggle/working" if IS_KAGGLE else "./work")).resolve()
FS2K_ROOT  = None            # set manually only if auto-discovery fails

TAG = "task4" if MODE == "full" else "task4_smoke"
DIRS = {k: WORK_DIR / k / TAG for k in ["checkpoints", "models", "figures", "results", "optuna", "cache"]}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

SEED, VAL_FRAC = 42, 0.15
AMP = True
NUM_WORKERS = min(4, os.cpu_count() or 2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
STYLE = list(SG.STYLE_NAMES)

USE_WANDB = True
WANDB_PROJECT = "genai-assignment1"
WANDB_ENTITY = None
WANDB_SECRET_NAME = "WANDB_API_KEY"
GROUP = "task4" if MODE == "full" else "task4-smoke"
print(f"MODE={MODE} | device={device} {torch.cuda.get_device_name(0) if device.type == 'cuda' else ''} | torch {torch.__version__}")
print(json.dumps(S))

In [ ]:
WANDB_OK = False
if USE_WANDB:
    try:
        import wandb
        if IS_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            wandb.login(key=UserSecretsClient().get_secret(WANDB_SECRET_NAME))
        WANDB_OK = True
    except Exception as e:
        print("W&B login error:", repr(e))

def wb_init(name, job_type, config=None, **kw):
    if not WANDB_OK:
        return None
    if wandb.run is not None:
        wandb.finish()
    return wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, group=GROUP, name=name, job_type=job_type,
                      tags=["task4", MODE], config=config or {}, **kw)

def wb_log_figs(run, figs: dict, step=None):
    if run is not None:
        run.log({k: wandb.Image(str(v)) for k, v in figs.items()})

print("W&B:", "on" if WANDB_OK else "off")
if USE_WANDB and not WANDB_OK:
    raise RuntimeError("W&B login failed — attach the WANDB_API_KEY secret to THIS notebook and turn Internet on. Nothing has been trained.")

## 1. FS2K: official split, stratified validation split, loading

In [ ]:
root = Path(FS2K_ROOT) if FS2K_ROOT else SG.find_fs2k(INPUT_ROOT)
train_all = SG.read_annotations(root, "train")
test_recs = SG.read_annotations(root, "test")
train_recs, val_recs = SG.stratified_val_split(train_all, VAL_FRAC, SEED)
if S["train_subset"]:
    train_recs = train_recs[:S["train_subset"]]
if S["eval_subset"]:
    val_recs, test_recs = val_recs[:S["eval_subset"]], test_recs[:S["eval_subset"]]
assert not {r["name"] for r in train_recs} & {r["name"] for r in val_recs}
assert not {r["name"] for r in train_all} & {r["name"] for r in test_recs}, "official train and test overlap!"
split_info = {"root": str(root), "seed": SEED, "val_fraction": VAL_FRAC, "stratified_by": "style",
              "train": [r["name"] for r in train_recs], "val": [r["name"] for r in val_recs], "test": [r["name"] for r in test_recs]}
(DIRS["results"] / "fs2k_split.json").write_text(json.dumps(split_info, indent=1))
cnt = pd.DataFrame({sp: pd.Series([r["style"] for r in rs]).value_counts().sort_index() for sp, rs in
                    [("official train", train_all), ("train", train_recs), ("val", val_recs), ("test (untouched)", test_recs)]})
cnt.index = [STYLE[i] for i in cnt.index]
print("FS2K root:", root); display(cnt)

t0 = time.time()
tr_p, tr_s = SG.load_pairs(train_recs, SG.JITTER, DIRS["cache"] / "train_142.npz")       # 142 -> random 128 crops
va_p, va_s = SG.load_pairs(val_recs, SG.IMG, DIRS["cache"] / "val_128.npz")
tr_style = np.array([r["style"] for r in train_recs]); va_style = np.array([r["style"] for r in val_recs])
print(f"loaded train {tr_p.shape} val {va_p.shape} in {time.time() - t0:.1f}s (test is loaded only for the final evaluation)")

def to_m11(a):   # uint8 NHWC -> float NCHW in [-1, 1]
    return torch.from_numpy(a.transpose(0, 3, 1, 2).copy()).float() / 127.5 - 1
va_x, va_y, va_st = to_m11(va_p), to_m11(va_s), torch.from_numpy(va_style)
# fixed validation photos for progress tracking (same photos every time, covering all 3 styles)
FIXED = [int(i) for s in range(3) for i in np.where(va_style == s)[0][:3]][:8]

fig, ax = plt.subplots(2, 6, figsize=(12, 4.4))
for j, i in enumerate([int(np.where(tr_style == s)[0][k]) for s in range(3) for k in range(2)]):
    ax[0, j].imshow(tr_p[i]); ax[1, j].imshow(tr_s[i]); ax[0, j].set_title(STYLE[tr_style[i]], fontsize=9)
for a in ax.flat:
    a.set_xticks([]); a.set_yticks([])
ax[0, 0].set_ylabel("photo"); ax[1, 0].set_ylabel("sketch"); fig.suptitle("FS2K training pairs (142×142 before random crop)")
fig.tight_layout(); fig.savefig(DIRS["figures"] / "dataset_pairs.png", dpi=130); plt.show()

In [ ]:
pre = []
def pf(name, ok, detail="", fatal=True):
    pre.append({"check": name, "passed": bool(ok), "fatal": fatal, "detail": str(detail)})
    print(("PASS " if ok else ("FAIL " if fatal else "WARN ")) + name + (f"  — {detail}" if detail else ""))

gpu = torch.cuda.is_available()
pf("GPU available", gpu, torch.cuda.get_device_name(0) if gpu else "Settings -> Accelerator -> GPU T4", fatal=(MODE == "full"))
wb_ok, wb_detail = False, "USE_WANDB is False"
if USE_WANDB:
    try:
        r = wb_init("preflight", "preflight", {"mode": MODE, "settings": S})
        r.log({"preflight/ok": 1}); wb_detail = r.url; r.finish(); wb_ok = True
    except Exception as e:
        wb_detail = repr(e)
pf("W&B login + test run", wb_ok or not USE_WANDB, wb_detail)
pf("FS2K official split sizes (1058 train / 1046 test)", len(train_all) == 1058 and len(test_recs) == 1046,
   f"{len(train_all)} train / {len(test_recs)} test", fatal=False)
pf("every annotation has its photo + sketch file", len(train_all) > 0 and len(test_recs) > 0, f"{len(train_all) + len(test_recs)} pairs found")
pf("validation split stratified by style", set(va_style) == set(tr_style), dict(zip(*np.unique(va_style, return_counts=True))))
METRICS = SG.Metrics(device)
pf("LPIPS available (perceptual metric used in the Optuna objective)", METRICS.lpips is not None,
   "" if METRICS.lpips is not None else getattr(METRICS, "lpips_error", ""), fatal=(MODE == "full"))
FID_OK = False
try:
    from torchmetrics.image.fid import FrechetInceptionDistance
    _f = FrechetInceptionDistance(feature=2048, normalize=True); del _f; FID_OK = True
    pf("FID (torchmetrics + Inception weights)", True)
except Exception as e:
    pf("FID (torchmetrics + Inception weights)", False, repr(e)[:150], fatal=False)
try:
    import onnx, onnxruntime
    pf("onnx + onnxruntime importable", True)
except Exception as e:
    pf("onnx + onnxruntime importable", False, repr(e))
free = shutil.disk_usage(WORK_DIR).free / 1e9
pf("free disk > 5 GB", free > 5, f"{free:.1f} GB")
pre_df = pd.DataFrame(pre)
bad = pre_df[pre_df.fatal & ~pre_df.passed]
assert bad.empty, "PRE-FLIGHT FAILED — nothing has been trained. Fix these and re-run:\n" + bad.to_string()
print(f"\nPRE-FLIGHT OK ({int(pre_df.passed.sum())}/{len(pre_df)} passed) — safe to train.")

## 2. Unit tests

In [ ]:
tests = []
def check(name, ok, detail=""):
    tests.append({"test": name, "passed": bool(ok), "detail": str(detail)})
    print(("PASS " if ok else "FAIL ") + name + (f"  — {detail}" if detail else ""))

# Pair-consistent augmentation: give the dataset IDENTICAL photo and sketch -> outputs must stay identical
dsx = SG.PairDataset(tr_p[:4], tr_p[:4].copy(), tr_style[:4], augment=True)
same = all(torch.equal(*dsx[i % 4][:2]) for i in range(40))
check("augmentation is identical for both images of a pair (40 random draws)", same)
x, y, s = dsx[0]
check("pair tensors (3,128,128) in [-1,1], style int", tuple(x.shape) == (3, 128, 128) and x.min() >= -1 and x.max() <= 1 and s.dtype == torch.int64)

G = SG.UNetGenerator(32, 8, 0.5).to(device); D = SG.PatchDiscriminator(32, 8).to(device)
G.apply(SG.init_weights); D.apply(SG.init_weights)
xb, yb, sb = (t.to(device) for t in (va_x[:4], va_y[:4], va_st[:4]))
out = G(xb, sb); lg = D(xb, out, sb)
check("G output (B,3,128,128) in [-1,1]; D patch logits (B,1,14,14)", tuple(out.shape) == (4, 3, 128, 128) and out.abs().max() <= 1 and tuple(lg.shape) == (4, 1, 14, 14))
check("style embedding exists in BOTH G and D", G.style_emb.num_embeddings == 3 and D.style_emb.num_embeddings == 3)
with torch.no_grad():
    for m in G.modules():
        if isinstance(m, SG.CondInstanceNorm):
            torch.nn.init.normal_(m.gamma.weight, 0, 0.3)
    G.eval(); d_style = (G(xb[:1], torch.tensor([0], device=device)) - G(xb[:1], torch.tensor([2], device=device))).abs().max().item()
    dl = (D(xb[:1], yb[:1], torch.tensor([0], device=device)) - D(xb[:1], yb[:1], torch.tensor([2], device=device))).abs().max().item()
check("changing the style changes G's output and D's judgement", d_style > 1e-3 and dl > 1e-4, f"|ΔG| {d_style:.3f}, |ΔD| {dl:.4f}")

G = SG.UNetGenerator(32, 8, 0.5).to(device); D = SG.PatchDiscriminator(32, 8).to(device); G.apply(SG.init_weights); D.apply(SG.init_weights)
ld = torch.utils.data.DataLoader(SG.PairDataset(tr_p[:16], tr_s[:16], tr_style[:16], augment=True), batch_size=8, shuffle=True)
og = torch.optim.Adam(G.parameters(), 2e-4, betas=(0.5, 0.999)); od = torch.optim.Adam(D.parameters(), 2e-4, betas=(0.5, 0.999))
st = SG.train_one_epoch(G, D, ld, og, od, device, 100.0, R.make_scaler(device, AMP), R.make_scaler(device, AMP), AMP)
check("one GAN epoch: all four losses finite and logged separately", all(np.isfinite(st[k]) for k in ("d_real", "d_fake", "g_adv", "g_l1")),
      {k: round(v, 3) for k, v in st.items()})
m = METRICS.per_image(((out[:2].detach().float().cpu() + 1) / 2).clamp(0, 1), (yb[:2].cpu() + 1) / 2)
check("validation metrics computed (L1, PSNR, SSIM" + (", LPIPS)" if "lpips" in m else ")"), all(np.isfinite(v).all() for v in m.values()))

import onnxruntime as ort
probe = DIRS["models"] / "_probe.onnx"
ex = SG.GeneratorExport(G.cpu().eval()).eval()
torch.onnx.export(ex, (torch.rand(1, 3, 128, 128), torch.tensor([1])), str(probe), input_names=["photo", "style"], output_names=["sketch"],
                  opset_version=17, dynamic_axes={"photo": {0: "batch"}, "style": {0: "batch"}, "sketch": {0: "batch"}}, dynamo=False)
o = ort.InferenceSession(str(probe), providers=["CPUExecutionProvider"]).run(None, {"photo": np.random.rand(3, 3, 128, 128).astype(np.float32),
                                                                                      "style": np.array([0, 1, 2], dtype=np.int64)})[0]
check("ONNX generator takes (photo, int64 style) and returns a [0,1] sketch", o.shape == (3, 3, 128, 128) and o.min() >= 0 and o.max() <= 1)
probe.unlink(); del G, D, og, od, ld
tests_df = pd.DataFrame(tests)
assert tests_df.passed.all(), f"UNIT TESTS FAILED:\n{tests_df[~tests_df.passed]}"
print(f"\nall {len(tests_df)} tests passed")

## 3. Optuna search (shorter trials)
* **Tuned:** generator lr, discriminator lr, batch size, base channel count (shared by G and D), dropout (first 3 decoder blocks), style-embedding dimension, λ_L1.
* **Objective (maximised):** 0.5·SSIM + 0.5·(1 − LPIPS) between generated and real validation sketches, averaged over the **last 3 epochs** of each trial. A GAN's quality fluctuates from epoch to epoch, so a single epoch would be an unreliable score.
* Each trial uses `trial_epochs` epochs with the pix2pix schedule (constant lr, then linear decay to 0). The chosen configuration is then retrained for the full schedule.
* The first trial is the standard pix2pix configuration (lr 2e-4 / 2e-4, λ_L1 = 100, dropout 0.5).

In [ ]:
SPACE = {
    "g_lr":         {"type": "float", "low": 5e-5, "high": 5e-4, "log": True},
    "d_lr":         {"type": "float", "low": 5e-5, "high": 5e-4, "log": True},
    "batch_size":   {"type": "categorical", "choices": [4, 8, 16]},
    "base_channels": {"type": "categorical", "choices": [32, 48, 64]},
    "dropout":      {"type": "float", "low": 0.0, "high": 0.5},
    "emb_dim":      {"type": "categorical", "choices": [8, 16, 32]},
    "lambda_l1":    {"type": "float", "low": 10.0, "high": 200.0, "log": True},
}
BASELINE = {"g_lr": 2e-4, "d_lr": 2e-4, "batch_size": 8, "base_channels": 64, "dropout": 0.5, "emb_dim": 16, "lambda_l1": 100.0}
(DIRS["results"] / "search_space.json").write_text(json.dumps(SPACE, indent=1))

def evaluate_val(G):
    fake = SG.generate(G, va_x, va_st, device)
    m = METRICS.per_image(fake, (va_y + 1) / 2)
    res = {f"val/{k}": float(np.mean(v)) for k, v in m.items()}
    for s in range(3):
        msk = va_style == s
        for k, v in m.items():
            res[f"val/{k}/style{s + 1}"] = float(np.mean(v[msk]))
    res["val/score"] = SG.val_score(m)
    return res

@torch.no_grad()
def sample_grid(G, ep, path):
    fake = SG.generate(G, va_x[FIXED], va_st[FIXED], device).numpy().transpose(0, 2, 3, 1)
    n = len(FIXED)
    fig, ax = plt.subplots(3, n, figsize=(1.6 * n, 5))
    for j, i in enumerate(FIXED):
        ax[0, j].imshow(va_p[i]); ax[1, j].imshow(va_s[i]); ax[2, j].imshow(fake[j]); ax[0, j].set_title(STYLE[va_style[i]], fontsize=8)
    for a in ax.flat:
        a.set_xticks([]); a.set_yticks([])
    ax[0, 0].set_ylabel("photo"); ax[1, 0].set_ylabel("real sketch"); ax[2, 0].set_ylabel(f"G, epoch {ep}")
    fig.suptitle(f"Fixed validation photos — epoch {ep}"); fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)
    return fake

def train_gan(hp, epochs, run=None, trial=None, sample_every=None, ckpt_dir=None, verbose=False):
    seed = SEED + (trial.number if trial else 0)
    torch.manual_seed(seed); np.random.seed(seed)
    G = SG.UNetGenerator(hp["base_channels"], hp["emb_dim"], hp["dropout"]).to(device)
    D = SG.PatchDiscriminator(hp["base_channels"], hp["emb_dim"]).to(device)
    G.apply(SG.init_weights); D.apply(SG.init_weights)
    loader = torch.utils.data.DataLoader(SG.PairDataset(tr_p, tr_s, tr_style, augment=True), batch_size=hp["batch_size"], shuffle=True,
                                         drop_last=True, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda",
                                         worker_init_fn=lambda w: np.random.seed(seed * 100 + w))
    og = torch.optim.Adam(G.parameters(), hp["g_lr"], betas=(0.5, 0.999)); od = torch.optim.Adam(D.parameters(), hp["d_lr"], betas=(0.5, 0.999))
    sg = torch.optim.lr_scheduler.LambdaLR(og, SG.linear_decay(epochs)); sd = torch.optim.lr_scheduler.LambdaLR(od, SG.linear_decay(epochs))
    scg, scd = R.make_scaler(device, AMP), R.make_scaler(device, AMP)
    hist, snapshots, best = [], {}, -1.0
    for ep in range(epochs):
        t0 = time.time()
        tr = SG.train_one_epoch(G, D, loader, og, od, device, hp["lambda_l1"], scg, scd, AMP)
        sg.step(); sd.step()
        ev = evaluate_val(G)
        rec = {"epoch": ep, "lr_g": sg.get_last_lr()[0], "lr_d": sd.get_last_lr()[0], "time_s": time.time() - t0,
               **{f"train/{k}": v for k, v in tr.items()}, **ev}
        hist.append(rec)
        log = dict(rec)
        if sample_every and (ep % sample_every == 0 or ep == epochs - 1):
            p = DIRS["figures"] / "progress_latest.png"
            snapshots[ep] = sample_grid(G, ep, p)
            if run:
                log["val_samples"] = wandb.Image(str(p), caption=f"epoch {ep}")
        if run:
            run.log(log)
        if ckpt_dir is not None and ev["val/score"] > best:
            best = ev["val/score"]
            torch.save({"G": G.state_dict(), "G_config": G.config, "hp": hp, "epoch": ep, "val": ev}, ckpt_dir / "task4_generator_bestval.pth")
        if verbose:
            print(f"ep {ep:3d} | D real {tr['d_real']:.3f} fake {tr['d_fake']:.3f} (acc {tr['d_acc_real']:.2f}/{tr['d_acc_fake']:.2f}) | "
                  f"G adv {tr['g_adv']:.3f} L1 {tr['g_l1']:.4f} | val SSIM {ev['val/ssim']:.4f}" +
                  (f" LPIPS {ev['val/lpips']:.4f}" if "val/lpips" in ev else "") + f" score {ev['val/score']:.4f} | {rec['time_s']:.0f}s")
            if ep == 0:
                print(f"   ≈ {rec['time_s'] * epochs / 60:.0f} min for {epochs} epochs")
        if trial:
            trial.report(ev["val/score"], ep)
            if trial.should_prune():
                raise optuna.TrialPruned()
    final_score = float(np.mean([h["val/score"] for h in hist[-3:]]))
    del loader
    return G, D, hist, snapshots, final_score

def objective(trial):
    hp = {}
    for k, s in SPACE.items():
        hp[k] = (trial.suggest_float(k, s["low"], s["high"], log=s.get("log", False)) if s["type"] == "float"
                 else trial.suggest_categorical(k, s["choices"]))
    run = wb_init(f"gan-trial-{trial.number:03d}", "gan-trial", {**hp, "trial": trial.number, "epochs": S["trial_epochs"]})
    state = "complete"
    try:
        G, D, hist, _, score = train_gan(hp, S["trial_epochs"], run, trial)
        trial.set_user_attr("val_ssim_last", hist[-1]["val/ssim"])
        if "val/lpips" in hist[-1]:
            trial.set_user_attr("val_lpips_last", hist[-1]["val/lpips"])
        del G, D
    except optuna.TrialPruned:
        state = "pruned"; raise
    except torch.cuda.OutOfMemoryError:
        state = "oom"; raise optuna.TrialPruned("OOM")
    finally:
        if run:
            run.summary.update({"state": state}); run.finish()
        gc.collect(); torch.cuda.empty_cache()
    return score

In [ ]:
STUDY = f"{TAG}_cgan"
DB = DIRS["optuna"] / f"{STUDY}.db"
if MODE == "smoke" and DB.exists():
    DB.unlink()
study = optuna.create_study(study_name=STUDY, storage=f"sqlite:///{DB}", direction="maximize", load_if_exists=True,
                            sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True, n_startup_trials=min(8, S["n_trials"])),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=max(1, S["trial_epochs"] // 3)))
if len(study.trials) == 0:
    study.enqueue_trial(BASELINE)
t0 = time.time()
done = lambda: len([t for t in study.trials if t.state in (optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED)])
while done() < S["n_trials"]:
    left = None if S["optuna_timeout"] is None else S["optuna_timeout"] - (time.time() - t0)
    if left is not None and left <= 0:
        print("time budget reached"); break
    study.optimize(objective, n_trials=1, timeout=left, gc_after_trial=True, catch=(FloatingPointError,))
print(f"study time {(time.time() - t0) / 60:.1f} min")
tdf = study.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs", "duration"))
tdf.to_csv(DIRS["results"] / "optuna_trials.csv", index=False)
states = tdf["state"].value_counts().to_dict()
BEST_HP = dict(study.best_params)
print("trial states:", states); print(f"best trial #{study.best_trial.number}: {study.best_value:.4f}", json.dumps(BEST_HP, indent=1))
display(tdf.sort_values("value", ascending=False).head(8))
figs = {}
from optuna.visualization import matplotlib as ovm
for nm, fn in [("history", ovm.plot_optimization_history), ("importance", ovm.plot_param_importances),
               ("slice", ovm.plot_slice), ("parallel", ovm.plot_parallel_coordinate)]:
    try:
        obj = fn(study); fig = (obj.flat[0] if hasattr(obj, "flat") else obj).figure
        fig.set_size_inches(12 if nm in ("slice", "parallel") else 7, 4.5)
        p = DIRS["figures"] / f"optuna_{nm}.png"; fig.savefig(p, dpi=140, bbox_inches="tight"); plt.show(); figs[f"optuna/{nm}"] = p
    except Exception as e:
        print(f"plot {nm} skipped: {e}")
plt.style.use("default")
(DIRS["results"] / "optuna_best.json").write_text(json.dumps({"best_trial": study.best_trial.number, "best_value": study.best_value,
    "best_params": BEST_HP, "user_attrs": study.best_trial.user_attrs, "n_trials": len(study.trials), "states": states,
    "trial_epochs": S["trial_epochs"], "search_space": SPACE, "baseline": BASELINE}, indent=1))
run = wb_init("gan-optuna-summary", "optuna-summary", {"search_space": SPACE, "trial_epochs": S["trial_epochs"]})
if run:
    run.summary.update({"best_value": study.best_value, "best_trial": study.best_trial.number, "states": states, **{f"best/{k}": v for k, v in BEST_HP.items()}})
    run.log({"optuna_trials": wandb.Table(dataframe=tdf.astype(str))}); wb_log_figs(run, figs)
    art = wandb.Artifact("task4-cgan-study", type="optuna-study"); art.add_file(str(DB)); run.log_artifact(art); run.finish()

## 4. Full retraining with the selected configuration
The final model is the generator **after the last epoch**: the learning rate has decayed to 0, as in pix2pix. The best-validation checkpoint is also saved, for reference.

In [ ]:
run = wb_init("gan-final", "gan-final", {**BEST_HP, "epochs": S["final_epochs"], "n_train": len(train_recs), "n_val": len(val_recs),
                                         "augmentation": "pair-identical random crop 142->128 + horizontal flip"})
t0 = time.time()
G, D, hist, snapshots, _ = train_gan(BEST_HP, S["final_epochs"], run, None, S["sample_every"], DIRS["checkpoints"], verbose=True)
print(f"training {(time.time() - t0) / 60:.1f} min")
FINAL_CKPT = DIRS["checkpoints"] / "task4_generator_final.pth"
torch.save({"G": G.state_dict(), "G_config": G.config, "D": D.state_dict(), "D_config": D.config, "hp": BEST_HP,
            "epoch": S["final_epochs"] - 1, "val": hist[-1]}, FINAL_CKPT)
h = pd.DataFrame(hist); h.to_csv(DIRS["results"] / "training_history.csv", index=False)

fig, ax = plt.subplots(1, 4, figsize=(19, 3.6))
ax[0].plot(h.epoch, h["train/d_real"], label="D real loss"); ax[0].plot(h.epoch, h["train/d_fake"], label="D fake loss"); ax[0].set_title("discriminator"); ax[0].legend()
ax[1].plot(h.epoch, h["train/g_adv"], label="G adversarial"); ax1b = ax[1].twinx(); ax1b.plot(h.epoch, h["train/g_l1"], "C1", label="G L1")
ax[1].set_title("generator (adv left, L1 right)"); ax[1].legend(loc="upper left"); ax1b.legend(loc="upper right")
ax[2].plot(h.epoch, h["train/d_acc_real"], label="D acc real"); ax[2].plot(h.epoch, h["train/d_acc_fake"], label="D acc fake"); ax[2].set_ylim(0, 1); ax[2].legend(); ax[2].set_title("discriminator patch accuracy")
for k, c in (("val/ssim", "C0"), ("val/lpips", "C3"), ("val/score", "k")):
    if k in h:
        ax[3].plot(h.epoch, h[k], color=c, label=k.split("/")[1])
ax[3].legend(); ax[3].set_title("validation")
for a in ax:
    a.set_xlabel("epoch")
fig.tight_layout(); fig.savefig(DIRS["figures"] / "training_curves.png", dpi=150); plt.show()

# progress figure: the same validation photos across training
eps = sorted(snapshots)[:: max(1, len(snapshots) // 6)][:6]
if (S["final_epochs"] - 1) not in eps:
    eps.append(S["final_epochs"] - 1)
fig, ax = plt.subplots(len(FIXED[:4]), len(eps) + 2, figsize=(1.7 * (len(eps) + 2), 1.8 * len(FIXED[:4])), squeeze=False)
for r, i in enumerate(FIXED[:4]):
    ax[r, 0].imshow(va_p[i]); ax[r, 1].imshow(va_s[i])
    for c, e in enumerate(eps):
        ax[r, c + 2].imshow(snapshots[e][r]); ax[0, c + 2].set_title(f"epoch {e}", fontsize=8)
ax[0, 0].set_title("photo", fontsize=8); ax[0, 1].set_title("real sketch", fontsize=8)
for a in ax.flat:
    a.set_xticks([]); a.set_yticks([])
fig.suptitle("Generator development on fixed validation photos"); fig.tight_layout()
fig.savefig(DIRS["figures"] / "training_progress.png", dpi=130); plt.show()
if run:
    wb_log_figs(run, {"figures/training_curves": DIRS["figures"] / "training_curves.png", "figures/training_progress": DIRS["figures"] / "training_progress.png"})
    art = wandb.Artifact("task4-generator-checkpoint", type="model", metadata=BEST_HP); art.add_file(str(FINAL_CKPT)); run.log_artifact(art)

## 5. Test evaluation (the official test set, used here for the first time)

In [ ]:
te_p, te_s = SG.load_pairs(test_recs, SG.IMG, DIRS["cache"] / "test_128.npz")
te_style = np.array([r["style"] for r in test_recs])
te_x, te_y, te_st = to_m11(te_p), to_m11(te_s), torch.from_numpy(te_style)
fake = SG.generate(G, te_x, te_st, device)
real01 = (te_y + 1) / 2
m = METRICS.per_image(fake, real01)
tdf_ = pd.DataFrame({"name": [r["name"] for r in test_recs], "style": [STYLE[s] for s in te_style], **m})
tdf_.to_csv(DIRS["results"] / "test_per_image.csv", index=False)
tab = tdf_.groupby("style")[list(m)].mean()
tab.loc["all"] = tdf_[list(m)].mean()
tab["n"] = list(tdf_.groupby("style").size()) + [len(tdf_)]

def fid_of(a01, b01):
    f = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
    for i in range(0, len(a01), 64):
        f.update(b01[i:i + 64].to(device), real=True); f.update(a01[i:i + 64].to(device), real=False)
    return float(f.compute())
if FID_OK:
    try:
        tab["FID"] = [fid_of(fake[te_style == s], real01[te_style == s]) for s in range(3)] + [fid_of(fake, real01)]
    except Exception as e:
        print("FID skipped:", repr(e)[:200])
print("Test metrics (lower L1/LPIPS/FID and higher PSNR/SSIM are better):"); display(tab.round(4))
tab.to_csv(DIRS["results"] / "test_metrics_by_style.csv")
try:
    (DIRS["results"] / "test_metrics_by_style.tex").write_text(tab.round(4).to_latex(float_format="%.4f"))
except Exception as e:
    print("LaTeX skipped:", e)

def grid(rows, path, title, cols=("photo", "real sketch", "generated")):
    fig, ax = plt.subplots(len(rows), len(cols), figsize=(2.2 * len(cols), 2.3 * len(rows)), squeeze=False)
    for r, row in enumerate(rows):
        for c, im in enumerate(row["ims"]):
            ax[r, c].imshow(np.clip(im, 0, 1)); ax[r, c].set_xticks([]); ax[r, c].set_yticks([])
        ax[r, 0].set_ylabel(row.get("label", ""), fontsize=8)
    for c, t in enumerate(cols):
        ax[0, c].set_title(t, fontsize=9)
    fig.suptitle(title); fig.tight_layout(); fig.savefig(path, dpi=130, bbox_inches="tight"); plt.show()
    return path

fk = fake.numpy().transpose(0, 2, 3, 1)
score_col = "lpips" if "lpips" in m else "ssim"
figs = {}
# 12 representative test examples: 4 per style, closest to the style's median score
rep = []
for s in range(3):
    idx = np.where(te_style == s)[0]
    v = m[score_col][idx]; rep += list(idx[np.argsort(np.abs(v - np.median(v)))[:4]])
figs["figures/test_examples"] = grid([{"ims": [te_p[i] / 255, te_s[i] / 255, fk[i]], "label": f"{STYLE[te_style[i]]}\nSSIM {m['ssim'][i]:.3f}"} for i in rep],
                                     DIRS["figures"] / "test_examples.png", "Test set: generated sketches in the annotated style")
# Failure cases: worst by the perceptual metric (or SSIM), one or two per style
worst = []
for s in range(3):
    idx = np.where(te_style == s)[0]; v = m[score_col][idx]
    worst += list(idx[np.argsort(-v if score_col == "lpips" else v)[:2]])
figs["figures/test_failures"] = grid([{"ims": [te_p[i] / 255, te_s[i] / 255, fk[i]], "label": f"{STYLE[te_style[i]]}\n{score_col} {m[score_col][i]:.3f}"} for i in worst],
                                     DIRS["figures"] / "test_failures.png", f"Failure cases (worst {score_col.upper()} per style)")
tdf_.iloc[worst].to_csv(DIRS["results"] / "failure_cases.csv", index=False)
# Style swap: the same photos rendered in all three styles
sw = [int(np.where(te_style == s)[0][k]) for s in range(3) for k in range(2)]
outs = [SG.generate(G, te_x[sw], torch.full((len(sw),), s), device).numpy().transpose(0, 2, 3, 1) for s in range(3)]
figs["figures/style_swap"] = grid([{"ims": [te_p[i] / 255, te_s[i] / 255] + [outs[s][j] for s in range(3)], "label": f"true: {STYLE[te_style[i]]}"}
                                   for j, i in enumerate(sw)], DIRS["figures"] / "style_swap.png",
                                  "Same photo, three requested styles", cols=("photo", "real sketch", *[f"G(x, {n})" for n in STYLE]))

## 6. Style consistency: does the generated sketch carry the requested style?
A small CNN is trained to recognise the three styles from **real** training sketches (validated on the real validation sketches).
It is then applied to (a) the real test sketches, which gives the upper bound; (b) generated test sketches in their annotated style; and (c) every test photo rendered in **each** of the three styles.
If the embedding controls the style, the classifier's prediction should follow the **requested** style.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
def to01(a): return torch.from_numpy(a.transpose(0, 3, 1, 2).copy()).float() / 255
crop = (SG.JITTER - SG.IMG) // 2
tr_sk01 = to01(tr_s[:, crop:crop + SG.IMG, crop:crop + SG.IMG])
torch.manual_seed(SEED)
sclf = C.CorruptionClassifier("16-32-64-128", 0.2, num_classes=3).to(device)
opt = torch.optim.AdamW(sclf.parameters(), 1e-3, weight_decay=1e-4)
dl = DataLoader(TensorDataset(tr_sk01, torch.from_numpy(tr_style)), batch_size=32, shuffle=True)
@torch.no_grad()
def style_pred(x01):
    sclf.eval(); return torch.cat([sclf(x01[i:i + 64].to(device)).argmax(1).cpu() for i in range(0, len(x01), 64)]).numpy()
for ep in range(S["style_epochs"]):
    sclf.train()
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        if np.random.rand() < 0.5:
            xb = xb.flip(3)
        loss = F.cross_entropy(sclf(xb), yb); opt.zero_grad(); loss.backward(); opt.step()
va_acc = (style_pred((va_y + 1) / 2) == va_style).mean()
print(f"style classifier: real validation sketches accuracy {va_acc:.3f}")
acc_real = (style_pred(real01) == te_style).mean()
acc_gen = (style_pred(fake) == te_style).mean()
from sklearn.metrics import confusion_matrix
req, prd = [], []
for s in range(3):
    o = SG.generate(G, te_x, torch.full((len(te_x),), s), device)
    req += [s] * len(o); prd += list(style_pred(o))
req, prd = np.array(req), np.array(prd)
acc_swap = (req == prd).mean()
cm = confusion_matrix(req, prd, labels=[0, 1, 2]); cmn = cm / cm.sum(1, keepdims=True)
style_tab = pd.DataFrame({"setting": ["real test sketches (upper bound)", "generated, annotated style", "generated, every photo x every style"],
                          "style accuracy": [acc_real, acc_gen, acc_swap]}).set_index("setting")
style_tab.loc["real validation sketches"] = va_acc
display(style_tab.round(3)); style_tab.to_csv(DIRS["results"] / "style_consistency.csv")
fig, ax = plt.subplots(figsize=(4.4, 3.9))
ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{cmn[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center", fontsize=8, color="white" if cmn[i, j] > .6 else "black")
ax.set_xticks(range(3)); ax.set_xticklabels(STYLE); ax.set_yticks(range(3)); ax.set_yticklabels(STYLE)
ax.set_xlabel("style recognised in generated sketch"); ax.set_ylabel("style requested"); ax.set_title(f"Style control (acc {acc_swap:.2f})")
fig.tight_layout(); fig.savefig(DIRS["figures"] / "style_control_confusion.png", dpi=150); plt.show()

if run:
    run.summary.update({**{f"test/{k}": float(v) for k, v in tab.loc["all"].items() if k != "n"},
                        "style/acc_real": float(acc_real), "style/acc_generated": float(acc_gen), "style/acc_swap": float(acc_swap)})
    run.log({"test/metrics_by_style": wandb.Table(dataframe=tab.reset_index()), "style/consistency": wandb.Table(dataframe=style_tab.reset_index())})
    wb_log_figs(run, {**figs, "figures/style_control_confusion": DIRS["figures"] / "style_control_confusion.png"})

## 7. ONNX export of the generator
Inputs: `photo` [B,3,128,128] float in [0,1] and `style` [B] int64 (0, 1, 2 = Style 1, 2, 3). Output: `sketch` [B,3,128,128] in [0,1]. The discriminator is a training-only component and is not exported.

In [ ]:
import onnx, onnxruntime as ort
ONNX_PATH = DIRS["models"] / "task4_face2sketch_generator.onnx"
G_cpu = SG.UNetGenerator(**G.config); G_cpu.load_state_dict({k: v.cpu() for k, v in G.state_dict().items()}); G_cpu.eval()
ex = SG.GeneratorExport(G_cpu).eval()
with torch.no_grad():
    torch.onnx.export(ex, (torch.rand(1, 3, 128, 128), torch.tensor([0])), str(ONNX_PATH), input_names=["photo", "style"], output_names=["sketch"],
                      opset_version=17, dynamic_axes={"photo": {0: "batch"}, "style": {0: "batch"}, "sketch": {0: "batch"}}, dynamo=False)
onnx.checker.check_model(onnx.load(str(ONNX_PATH)))
sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
xs = ((te_x[:32] + 1) / 2)
parity = {}
for s in range(3):
    st_ = torch.full((len(xs),), s)
    with torch.no_grad():
        ref = ex(xs, st_).numpy()
    o = sess.run(None, {"photo": xs.numpy(), "style": st_.numpy()})[0]
    parity[STYLE[s]] = {"max_abs_diff": float(np.abs(o - ref).max()), "mean_abs_diff": float(np.abs(o - ref).mean())}
    parity[STYLE[s]]["passed"] = parity[STYLE[s]]["max_abs_diff"] < 1e-4
ts = []
for _ in range(10):
    sess.run(None, {"photo": xs[:1].numpy(), "style": np.array([0], dtype=np.int64)})
for _ in range(50 if MODE == "full" else 10):
    t0 = time.perf_counter(); sess.run(None, {"photo": xs[:1].numpy(), "style": np.array([1], dtype=np.int64)}); ts.append((time.perf_counter() - t0) * 1000)
latency = {"median_ms": float(np.median(ts)), "p90_ms": float(np.percentile(ts, 90)), "batch": 1}
size_mb = ONNX_PATH.stat().st_size / 1e6
print(json.dumps({"parity": parity, "cpu_latency": latency, "size_mb": round(size_mb, 1)}, indent=1))
assert all(v["passed"] for v in parity.values()), "ONNX parity failed"
sidecar = {"task": "face_to_sketch", "onnx_file": ONNX_PATH.name, "opset": 17, "styles": {"0": "Style 1", "1": "Style 2", "2": "Style 3"},
           "inputs": {"photo": "[batch,3,128,128] float32 RGB in [0,1], NCHW", "style": "[batch] int64 in {0,1,2}"},
           "output": {"sketch": "[batch,3,128,128] float32 RGB in [0,1]"},
           "preprocess": "PIL convert('RGB') (alpha -> white) -> resize 128x128 BICUBIC -> /255",
           "generator_config": G.config, "hyperparameters": BEST_HP, "onnx_parity": parity, "cpu_latency_ms": latency,
           "test_metrics": {k: float(v) for k, v in tab.loc["all"].items()}}
(DIRS["models"] / "task4_face2sketch_generator.json").write_text(json.dumps(sidecar, indent=1))
if run:
    run.summary.update({"onnx/max_abs_diff": max(v["max_abs_diff"] for v in parity.values()), "onnx/latency_ms": latency["median_ms"], "onnx/size_mb": size_mb})
    art = wandb.Artifact("task4-face2sketch-onnx", type="model", metadata=sidecar)
    art.add_file(str(ONNX_PATH)); art.add_file(str(DIRS["models"] / "task4_face2sketch_generator.json")); run.log_artifact(art); run.finish()

## 8. Summary

In [ ]:
summary = {"mode": MODE, "fs2k_root": str(root), "n": {"train": len(train_recs), "val": len(val_recs), "test": len(test_recs)},
           "best_trial": study.best_trial.number, "best_hp": BEST_HP, "optuna_states": states,
           "test_all": {k: float(v) for k, v in tab.loc["all"].items()},
           "style_consistency": {"real": float(acc_real), "generated": float(acc_gen), "swap": float(acc_swap)},
           "onnx_parity_passed": all(v["passed"] for v in parity.values()), "onnx_size_mb": round(size_mb, 1),
           "unit_tests_passed": int(tests_df.passed.sum())}
(DIRS["results"] / "summary.json").write_text(json.dumps(summary, indent=1))
print(json.dumps(summary, indent=1))
print("\nOutputs:")
for d in DIRS.values():
    if d.name == "cache" or d.parent.name == "cache":
        continue
    for p in sorted(d.rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(WORK_DIR)}  ({p.stat().st_size / 1e6:.2f} MB)")
print("\nSMOKE TEST PASSED — switch MODE to 'full' and Save & Run All." if MODE == "smoke" else "\nTASK 4 COMPLETE.")